Importing Packages

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

Reading Dataframe

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Space Titanic/train.csv')
df.head(5)

Exploring Data

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

Visualizing Features

In [ ]:
sns.countplot(data=df, x='CryoSleep', hue='Transported')
plt.show()

In [ ]:

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(8,4), dpi=100)

sns.countplot(data=df, x='HomePlanet', hue='Transported', ax=ax[0])
ax[0].set_title('HomePlanet')

df['From Earth'] = (df['HomePlanet']=='Earth')
sns.countplot(data=df, x='From Earth', hue='Transported', ax=ax[1])
ax[1].set_title('Is From Earth')

fig.tight_layout()
plt.show()


In [ ]:
sns.histplot(data=df, x='Age', hue='Transported')
plt.show()

In [ ]:
money_spent = df[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].sum(axis=1)
did_spend = (money_spent==0)

sns.countplot(x=did_spend, hue=df['Transported'])

plt.show()

In [ ]:
df['Id'] = df['PassengerId'].str[-2:].astype(int)
df['Group'] = df['PassengerId'].str[:4].astype(int)

fig, ax = plt.subplots(figsize=(8,6))

sns.histplot(data=df, x='Group', hue='Transported', bins=200)
ax.set_title('Passenger Group')

plt.show()

In [ ]:
from collections import Counter

group_counter = np.vectorize(Counter(df['Group']).get)
df['GroupSize'] = group_counter(df['Group'])
df['Alone'] = (df['GroupSize']==1)

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(8,4), dpi=100)

sns.countplot(data=df, x='GroupSize', hue='Transported', ax=ax[0])
ax[0].set_title('Group Size')

sns.countplot(data=df, x='Alone', hue='Transported', ax=ax[1])
ax[1].set_title('Is Alone')

fig.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

model_oe = OrdinalEncoder()

df[['CabinDeck', 'CabinNumber', 'CabinSide']] = df['Cabin'].str.split('/', expand=True)
df['CabinNumber'] = df['CabinNumber'].astype(float)

df['CabinDeck'] = model_oe.fit_transform(df[['CabinDeck']])
df['CabinSide'] = model_oe.fit_transform(df[['CabinSide']])

df2 = df.loc[::30, :].dropna()
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection='3d')


ax.scatter3D(df2['CabinDeck'], df2['CabinNumber'], df2['CabinSide'], c=df2['Transported'])

Cleaning and Seperating Columns

In [ ]:
from collections import Counter

df['Id'] = df['PassengerId'].str[-2:].astype(int)
df['Group'] = df['PassengerId'].str[:4].astype(int)

group_counter = np.vectorize(Counter(df['Group']).get)
df['GroupSize'] = group_counter(df['Group'])
df['Alone'] = (df['GroupSize']==1)

df['From Earth'] = (df['HomePlanet']=='Earth')

df[['First Name','Last Name']] = df['Name'].str.split(' ', expand=True)

df.loc[df['Name'].str.split().str.len() == 2, 'LastNameLength'] = df['Last Name'].str.len()

counter = np.vectorize(Counter(df['Last Name']).get)
df['FamilySize'] = counter(df['Last Name'])

df[['CabinDeck', 'CabinNumber', 'CabinSide']] = df['Cabin'].str.split('/', expand=True)
df['CabinNumber'] = df['CabinNumber'].astype(float)

#'First Name','Last Name','LastNameLength', --> Dropped
df = df.loc[:,['Group','Id','HomePlanet','From Earth', 'GroupSize','Alone','FamilySize','CryoSleep','CabinDeck',
               'CabinNumber','CabinSide','VIP','Age','Destination','RoomService','FoodCourt',
               'ShoppingMall','Spa','VRDeck','Transported']]


Encoding Data to Work With

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

model_oe = OrdinalEncoder()
for column in df[['HomePlanet','CryoSleep','CabinDeck','CabinSide','VIP','Destination','Transported']].columns:
  print(f"Encoding {column}")
  new_column = model_oe.fit_transform(df[[column]])
  df[column] = new_column


Imputing Data with Iterative Imputer

In [ ]:
### TAKES ABOUT 3 MINUTES ###
# Imputes by focusing on one feature and looking at all the other ones, then moves on to next feature #

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

X = df.drop(['Transported'], axis=1)

features = X.columns.tolist()
print(len(features), features)

imputer = IterativeImputer(estimator=RandomForestRegressor(n_estimators=25), max_iter=25, random_state=42)

imputer.fit(df[features])
imputed_X = imputer.transform(df[features])

df[features] = imputed_X

Adding Some More Features (After Imputation, makes these features better)

In [ ]:
df['Money Spent'] = df[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].sum(axis=1)
df['Zero Spent'] = (df['Money Spent']==0)

Exploring Correlations

In [ ]:
df.corr()['Transported'].abs().sort_values(ascending=False)

Preparing and Splitting Data

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score

X = df.drop(['Transported'], axis=1)
y = df['Transported']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

X_train.shape, X_test.shape, y_train.shape, y_test.shape
                                                    


Feature Selection

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import RFECV
from xgboost import XGBClassifier as XGB

base_xgb = XGB()

min_features_to_select = 5
rfecv = RFECV(estimator=base_xgb, step=1, cv=StratifiedKFold(2), scoring="accuracy", min_features_to_select=min_features_to_select, verbose=3)
rfecv.fit(X_train, y_train)

print("Optimal number of features : %d" % rfecv.n_features_)

plt.figure()
plt.xlabel("Number of features selected")
plt.ylabel("Cross validation score (accuracy)")
plt.plot(
    range(min_features_to_select, len(rfecv.cv_results_['mean_test_score']) + min_features_to_select),
    rfecv.cv_results_['mean_test_score'],
)
plt.show()

features_selected = X_train.loc[:, list(rfecv.support_)].columns.tolist()
features_selected

Comparing Models

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier as XGB
from sklearn.svm import SVC

classifiers = [
    RandomForestClassifier(n_estimators=100),
    AdaBoostClassifier(),
    GradientBoostingClassifier(),
    #LogisticRegression(max_iter=1000),
    #KNeighborsClassifier(3),
    #MLPClassifier(alpha=1, max_iter=1000),
    #DecisionTreeClassifier(max_depth=5),
    #MultinomialNB(),
    #GaussianNB(),
    SVC(),
    XGB()
]

for clf in classifiers:
    clf.fit(X_train, y_train)
    score = np.mean(cross_val_score(clf, X_test, y_test, cv=5))
    name = clf.__class__.__name__
    print(name, score)



Evaluating Model

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, recall_score, precision_score
from sklearn.model_selection import cross_val_score

GBC = GradientBoostingClassifier(learning_rate=0.03, max_depth=8, n_estimators=100, subsample=0.5)
GBC.fit(X_train.loc[:, features_selected], y_train)

predictions = GBC.predict(X_test.loc[:, features_selected])

mat = confusion_matrix(y_test, predictions, normalize='true')
sns.heatmap(mat, square=True, annot=True, cbar=True)
plt.xlabel('predicted value')
plt.ylabel('true value')
plt.show()

print("Accuracy Score:", accuracy_score(y_test, predictions))
print("Precision Score:", precision_score(y_test, predictions))
print("Recall/Sensitity Score:", recall_score(y_test, predictions)) 
print("F1 Score:", f1_score(y_test, predictions))

Parameter Tuning

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

xgb_model = XGBClassifier(verbosity=0)


parameters = {'nthread':[1,2,3,4,10],
              'objective':['binary:logistic'],
              'learning_rate': [0.01],
              'max_depth': [1,2,3,5,8,10],
              'min_child_weight': [1,2,3,10,12],
              'silent': [0],
              'subsample': [0.1,0.2,0.5,0.8],
              'colsample_bytree': [0.01, 0.1, 0.2, 0.5],
              'n_estimators': [1000],
              'missing':[-999],
              'seed': [1337]} # For Tuning

parameters = {'nthread':[1],
              'objective':['binary:logistic'],
              'learning_rate': [0.01],
              'max_depth': [8],
              'min_child_weight': [3],
              'silent': [1],
              'subsample': [0.8],
              'colsample_bytree': [0.5],
              'n_estimators': [1000],
              'missing':[-999],
              'seed': [1337]}

xgb_model = GridSearchCV(estimator=xgb_model, param_grid=parameters, n_jobs=1, 
                         cv=StratifiedKFold(n_splits=5, shuffle=True), 
                         scoring='f1', refit=True, verbose=3)

xgb_model.fit(X_train, y_train)

print(xgb_model.best_score_)
xgb_model.best_params_


Eavluating Tuned Model

In [ ]:
from xgboost import XGBClassifier

final_model = XGBClassifier(verbosity=0, nthread=1,
                            objective='binary:logistic',
                            learning_rate=0.01,
                            max_depth=8,
                            min_child_weight=3,
                            silent=1,
                            subsample=0.8,
                            colsample_bytree=0.5,
                            n_estimators=1000,
                            missing=-999)

final_model.fit(X_train.loc[:, features_selected], y_train)
predictions = final_model.predict(X_test.loc[:, features_selected])

mat = confusion_matrix(y_test, predictions, normalize='true')
sns.heatmap(mat, square=True, annot=True, cbar=True)
plt.xlabel('predicted value')
plt.ylabel('true value')
plt.show()

print("Accuracy Score:", accuracy_score(y_test, predictions))
print("Precision Score:", precision_score(y_test, predictions))
print("Recall/Sensitity Score:", recall_score(y_test, predictions)) 
print("F1 Score:", f1_score(y_test, predictions))


Creating Pipeline

In [ ]:
from collections import Counter

def process_data(X):
  X['Id'] = X['PassengerId'].str[-2:].astype(int)
  X['Group'] = X['PassengerId'].str[:4].astype(int)

  group_counter = np.vectorize(Counter(X['Group']).get)
  X['GroupSize'] = group_counter(X['Group'])
  X['Alone'] = (X['GroupSize']==1)

  X[['First Name','Last Name']] = X['Name'].str.split(' ', expand=True)

  X.loc[df['Name'].str.split().str.len() == 2, 'LastNameLength'] = X['Last Name'].str.len()

  counter = np.vectorize(Counter(X['Last Name']).get)
  X['FamilySize'] = counter(X['Last Name'])

  X[['CabinDeck', 'CabinNumber', 'CabinSide']] = X['Cabin'].str.split('/', expand=True)
  X['CabinNumber'] = X['CabinNumber'].astype(float)

  #'First Name','Last Name','LastNameLength', --> Dropped
  X = X.loc[:,['Group','Id','HomePlanet','GroupSize','Alone','FamilySize','CryoSleep','CabinDeck',
                'CabinNumber','CabinSide','VIP','Age','Destination','RoomService','FoodCourt',
                'ShoppingMall','Spa','VRDeck']]
  return X

from sklearn.preprocessing import OrdinalEncoder

def encode_data(X):
  model_oe = OrdinalEncoder()
  for column in X[['HomePlanet','CryoSleep','CabinDeck','CabinSide','VIP','Destination']].columns:
    new_column = model_oe.fit_transform(X[[column]])
    X[column] = new_column
  return X

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

def impute_data(X):
  features = X.columns.tolist()

  imputer = IterativeImputer(estimator=RandomForestRegressor(n_estimators=25), max_iter=25, random_state=42)

  imputer.fit(X[features])
  imputed_X = imputer.transform(X[features])

  X[features] = imputed_X
  return X

def add_features(X):
  X['Money Spent'] = X[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].sum(axis=1)
  X['Zero Spent'] = (X['Money Spent']==0)
  return X



In [ ]:

final_model = XGBClassifier(verbosity=0, nthread=1, objective='binary:logistic', learning_rate=0.01, max_depth=8, silent=1,
                            min_child_weight=3, subsample=0.8, colsample_bytree=0.5, n_estimators=1000, missing=-999)

feature_selection = RFECV(estimator=final_model, step=1, cv=StratifiedKFold(2), scoring="accuracy", min_features_to_select=5)



In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

process_data_transformer = FunctionTransformer(process_data)
encoding_data_transformer = FunctionTransformer(encode_data)
imputing_data_transformer = FunctionTransformer(impute_data)
adding_features_transformer = FunctionTransformer(add_features)

pipe = Pipeline(steps=[
    ('process_data', process_data_transformer),
    ('encode_data', encoding_data_transformer),
    ('impute_data', imputing_data_transformer),
    ('add_features', adding_features_transformer),
    ('feature_selection', feature_selection),
    ('model', final_model)
])

df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Space Titanic/train.csv')
X = df.drop(['Transported'], axis=1)
y = df['Transported']

pipe.fit(X, y)

In [ ]:
### Evaluating Pipeline with Itself to get a better idea of using full data set ###
predictions = pipe.predict(X)

mat = confusion_matrix(y, predictions, normalize='true')
sns.heatmap(mat, square=True, annot=True, cbar=True)
plt.xlabel('predicted value')
plt.ylabel('true value')
plt.show()

print("Accuracy Score:", accuracy_score(y, predictions))
print("Precision Score:", precision_score(y, predictions))
print("Recall/Sensitity Score:", recall_score(y, predictions)) 
print("F1 Score:", f1_score(y, predictions))



Final Submission File

In [ ]:
submission = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Space Titanic/test.csv').loc[:,['PassengerId']]
final_X = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Space Titanic/test.csv')
final_X.head()

In [ ]:
submission['Transported'] = pipe.predict(final_X).astype(bool)

In [ ]:
submission.to_csv('/content/drive/MyDrive/Kaggle Competitions/Space Titanic/submission.csv', index=False)